# ChlorA Heatmap — Two-Stage Environmental Model

**Architecture:**
1. **Stage 1 — Environment Model**: Predicts 10 oceanographic features (`T_degC`, `Salnty`, `O2ml_L`, etc.) from `Lat_Dec` + `Lon_Dec`.
2. **Stage 2 — ChlorA Model**: Predicts `ChlorA` from those 10 features.

**Heatmap sliders**: Apply a ±% multiplier to each spatially-estimated feature before passing into the ChlorA model — simulating environmental perturbations (e.g. "what if temperature rose 15%?").

In [ ]:
# Install dependencies if needed
%pip install -q scikit-learn plotly ipywidgets pandas numpy global-land-mask anywidget

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import cross_val_score

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# ── Load data ──────────────────────────────────────────────────────────────────
df = pd.read_csv('Finalized_CalCOFI_Phytoplankton.csv')

SPATIAL_FEATURES  = ['Lat_Dec', 'Lon_Dec']
ENV_FEATURES      = [
    'T_degC', 'Salnty', 'O2ml_L', 'O2Sat',
    'PO4uM', 'SiO3uM', 'NO3uM', 'Cloud_Amt', 'NP_Ratio', 'Phaeop'
]
TARGET = 'ChlorA'

all_cols = SPATIAL_FEATURES + ENV_FEATURES + [TARGET]
df_clean = df[all_cols].dropna()

print(f"Samples after dropping NaN: {len(df_clean):,}")
print(f"ChlorA range: {df_clean[TARGET].min():.3f} – {df_clean[TARGET].max():.3f} μg/L  "
      f"|  mean: {df_clean[TARGET].mean():.3f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE 1 — Predict ENV_FEATURES from Lat/Lon
# Uses MultiOutputRegressor wrapping a Random Forest
# ══════════════════════════════════════════════════════════════════════════════
X_spatial = df_clean[SPATIAL_FEATURES]
Y_env     = df_clean[ENV_FEATURES]

print("Training Stage 1: Lat/Lon → Environmental Features...")
stage1_model = MultiOutputRegressor(
    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    n_jobs=-1
)
stage1_model.fit(X_spatial, Y_env)
print("Stage 1 training complete.")

# Quick per-feature R² on training data as a sanity check
stage1_preds = stage1_model.predict(X_spatial)
from sklearn.metrics import r2_score
print("\nStage 1 per-feature R² (train):")
for feat, r2 in zip(ENV_FEATURES, [r2_score(Y_env[f], stage1_preds[:, i])
                                     for i, f in enumerate(ENV_FEATURES)]):
    bar = '█' * int(max(r2, 0) * 40)
    print(f"  {feat:12s}  R²={r2:.3f}  {bar}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE 2 — Predict ChlorA from ENV_FEATURES
# Compare RF vs Gradient Boosting; pick the better one
# ══════════════════════════════════════════════════════════════════════════════
X_env = df_clean[ENV_FEATURES]
y     = df_clean[TARGET]

print("Evaluating Stage 2 models (5-fold CV)...")
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf_rmse = -cross_val_score(rf, X_env, y, cv=5,
                            scoring='neg_root_mean_squared_error',
                            n_jobs=-1).mean()

gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                max_depth=5, random_state=42)
gb_rmse = -cross_val_score(gb, X_env, y, cv=5,
                            scoring='neg_root_mean_squared_error').mean()

print(f"  Random Forest  CV-RMSE: {rf_rmse:.4f}")
print(f"  Gradient Boost CV-RMSE: {gb_rmse:.4f}")

if rf_rmse <= gb_rmse:
    stage2_model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    model_name = "Random Forest"
else:
    stage2_model = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                              max_depth=5, random_state=42)
    model_name = "Gradient Boosting"

print(f"\n→ Selected: {model_name}")
stage2_model.fit(X_env, y)
print("Stage 2 training complete.")

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from global_land_mask import globe

# ── 1. Build ocean grid ────────────────────────────────────────────────────────
LAT_MIN, LAT_MAX = 29.5, 35.5
LON_MIN, LON_MAX = -122.5, -116.5
RESOLUTION = 0.1

lat_pts = np.arange(LAT_MIN, LAT_MAX + RESOLUTION, RESOLUTION)
lon_pts = np.arange(LON_MIN, LON_MAX + RESOLUTION, RESOLUTION)
lon_grid, lat_grid = np.meshgrid(lon_pts, lat_pts)

is_on_land = globe.is_land(lat_grid, lon_grid)
grid_lats  = lat_grid[~is_on_land].ravel()
grid_lons  = lon_grid[~is_on_land].ravel()

# ── 2. Stage 1: estimate baseline environmental values for every grid cell ─────
grid_spatial_df  = pd.DataFrame({'Lat_Dec': grid_lats, 'Lon_Dec': grid_lons})
baseline_env     = stage1_model.predict(grid_spatial_df)          # shape (n_cells, 10)
baseline_env_df  = pd.DataFrame(baseline_env, columns=ENV_FEATURES)

# ── 3. Prediction function — apply % multipliers then run Stage 2 ──────────────
def predict_grid(**pct_adjustments):
    """
    pct_adjustments: {feature_name: float} where value is a % change, e.g. +15 means +15%.
    Applies the multiplier to the spatially-estimated baseline, then predicts ChlorA.
    """
    env_adjusted = baseline_env_df.copy()
    for feat, pct in pct_adjustments.items():
        env_adjusted[feat] = baseline_env_df[feat] * (1 + pct / 100.0)
    preds = stage2_model.predict(env_adjusted[ENV_FEATURES])
    return np.clip(preds, 0, None)

init_preds  = predict_grid(**{feat: 0.0 for feat in ENV_FEATURES})
chlora_p95  = float(np.percentile(y, 95))

# ── 4. Plotly FigureWidget ─────────────────────────────────────────────────────
fig = go.FigureWidget(data=[go.Densitymapbox(
    lat=grid_lats,
    lon=grid_lons,
    z=init_preds,
    radius=12,
    colorscale='Viridis',
    reversescale=False,
    colorbar=dict(title='ChlorA<br>(μg/L)', thickness=16),
    zmin=0,
    zmax=chlora_p95,
    hovertemplate='Lat: %{lat:.2f}<br>Lon: %{lon:.2f}<br>Predicted ChlorA: %{z:.3f} μg/L<extra></extra>',
)])

fig.update_layout(
    mapbox_style="white-bg",
    mapbox_layers=[{
        "below": 'traces',
        "sourcetype": "raster",
        "source": ["https://basemaps.cartocdn.com/light_all/{z}/{x}/{y}.png"]
    }],
    mapbox_center={"lat": 32.5, "lon": -119.5},
    mapbox_zoom=5.5,
    title=dict(
        text="Ocean-Only Predicted Chlorophyll-A  ·  Two-Stage Environmental Model",
        x=0.5, font=dict(size=13)
    ),
    height=620,
    margin=dict(l=0, r=0, t=45, b=0),
)

# ── 5. Percentage sliders ──────────────────────────────────────────────────────
sliders = {}
for feat in ENV_FEATURES:
    sliders[feat] = widgets.FloatSlider(
        value=0.0,
        min=-50.0,
        max=50.0,
        step=1.0,
        description=f"{feat} %",
        continuous_update=False,
        readout_format='+.0f',
        style={'description_width': '130px'},
        layout=widgets.Layout(width='520px'),
    )

reset_btn    = widgets.Button(description="Reset All",
                               button_style='warning',
                               layout=widgets.Layout(width='120px', margin='4px 0 8px 0'))
status_label = widgets.Label(value="Sliders adjust each feature as % of its spatially-estimated baseline.")

def on_change(change):
    status_label.value = "Computing..."
    pct_vals = {feat: sliders[feat].value for feat in ENV_FEATURES}
    preds    = predict_grid(**pct_vals)
    with fig.batch_update():
        fig.data[0].z = preds
    status_label.value = "Sliders adjust each feature as % of its spatially-estimated baseline."

def on_reset(b):
    for s in sliders.values():
        s.value = 0.0

for s in sliders.values():
    s.observe(on_change, names='value')
reset_btn.on_click(on_reset)

slider_box = widgets.VBox(
    [
        widgets.HTML("<b style='font-size:13px'>Environmental perturbations (% change from spatial baseline):</b>"),
        status_label,
        reset_btn,
    ] + list(sliders.values()),
    layout=widgets.Layout(padding='8px 12px')
)

display(widgets.VBox([fig, slider_box]))